# Assignment 1: Student Performance Prediction

**Goal:** Use *Linear Regression* on the Student Habits & Performance dataset to predict a student's `Exam_Score` based on their daily habits (study hours, sleep, social media use, attendance, etc).

**Requirements covered in this notebook:**
- Load and explore the dataset
- Clean the data (handle missing values, remove unneeded columns)
- Encode categorical (text) columns into numbers so the model can use them
- Split data into features (X) and target (y)
- Split data into training and testing sets
- Train a Linear Regression model
- Evaluate the model using MAE, MSE, RMSE, and R² Score
- Compare actual vs predicted exam scores

**Target column:** `exam_score`
**Feature columns:** all other numeric/encoded columns

### Install pandas
Installing the pandas library, which we use to load and work with the dataset as a table (DataFrame).

In [85]:
# Install the pandas library (skips install if already present)
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Load the dataset
Importing pandas and reading the CSV file into a DataFrame called `data`, so we can explore and clean it.

In [86]:
# Import pandas and give it the short name 'pd'
import pandas as pd
# Read the CSV file into a DataFrame called 'data'
data = pd.read_csv("student_habits_performance.csv")

### Preview the data
Looking at the first 5 rows to see what the columns and values look like.

In [87]:
# Show the first 5 rows of the dataset
data.head()

,student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,S1000,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,S1001,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,S1002,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,S1003,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,S1004,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


### Check dataset size
Checking how many rows (students) and columns (features) the dataset has.

In [88]:
# Show number of rows and columns as (rows, columns)
data.shape

(1000, 16)

### Check column types and missing values
Looking at each column's data type (number vs text) and how many non-missing values it has.

In [89]:
# Show column names, data types, and non-null counts
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   student_id                     1000 non-null   str    
 1   age                            1000 non-null   int64  
 2   gender                         1000 non-null   str    
 3   study_hours_per_day            1000 non-null   float64
 4   social_media_hours             1000 non-null   float64
 5   netflix_hours                  1000 non-null   float64
 6   part_time_job                  1000 non-null   str    
 7   attendance_percentage          1000 non-null   float64
 8   sleep_hours                    1000 non-null   float64
 9   diet_quality                   1000 non-null   str    
 10  exercise_frequency             1000 non-null   int64  
 11  parental_education_level       909 non-null    str    
 12  internet_quality               1000 non-null   str    
 13  

### Check a categorical column's values
Looking at how many students fall into each category of `parental_education_level`, since this column had missing values (from the info() output above).

In [90]:
# Count how many times each category appears in parental_education_level
data.parental_education_level.value_counts()

parental_education_level
High School    392
Bachelor       350
Master         167
Name: count, dtype: int64

### Fill missing values
Filling the missing values in `parental_education_level` with the most common value (the mode), so we don't lose any rows.

In [91]:
# Fill missing values in parental_education_level with the most frequent category (mode)
data.parental_education_level = data["parental_education_level"].fillna(
    data["parental_education_level"].mode()[0],
    inplace=True
)

C:\Users\Muhammad Shan\AppData\Local\Temp\ipykernel_7244\2296572197.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data.parental_education_level = data["parental_education_level"].fillna(


### Confirm no missing values remain
Checking every column for missing (null) values to make sure the fill worked.

In [92]:
# Count missing values in each column (should all be 0 now)
data.isnull().sum()

student_id                       0
age                              0
gender                           0
study_hours_per_day              0
social_media_hours               0
netflix_hours                    0
part_time_job                    0
attendance_percentage            0
sleep_hours                      0
diet_quality                     0
exercise_frequency               0
parental_education_level         0
internet_quality                 0
mental_health_rating             0
extracurricular_participation    0
exam_score                       0
dtype: int64

### Check for duplicate rows
Making sure there are no exact duplicate student records in the dataset.

In [93]:
# Count how many duplicate rows exist in the dataset
data.duplicated().sum()

np.int64(0)

### Remove the ID column
Dropping `student_id` because it's just a unique label for each student, not something that can help predict exam scores.

In [94]:
# Remove the student_id column since it has no predictive value
data = data.drop(["student_id"],axis=1)

### Preview after dropping student_id
Confirming the column was removed.

In [95]:
# Show the first 5 rows again after dropping student_id
data.head()

,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


### Convert gender to numbers
Machine learning models need numbers, not text, so we manually map each gender category to a number (Male=0, Female=1, Other=2).

In [96]:
# Convert the gender column from text to numbers using a manual mapping
data["gender"] = data["gender"].map({"Male" : 0, "Female" : 1, "Other" : 2})

### Preview after encoding gender
Confirming gender is now numeric.

In [97]:
# Show the first 5 rows to confirm gender is now numeric
data.head()

,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,23,1,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,20,1,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,21,0,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,23,1,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,19,1,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


### Check unique gender values
Making sure only the expected numbers (0, 1, 2) exist after the mapping.

In [98]:
# List the unique values in the gender column
data["gender"].unique()

array([1, 0, 2])

### Count each gender value
Seeing how many students fall into each gender category after encoding.

In [99]:
# Count how many rows have each gender value
data["gender"].value_counts(dropna=False)

gender
1    481
0    477
2     42
Name: count, dtype: int64

### Install scikit-learn
Installing the scikit-learn library, which provides the encoding tools, model, and evaluation metrics we'll use next.

In [100]:
# Install the scikit-learn library (skips install if already present)
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Preview data before encoding remaining categorical columns
Checking which text columns are still left to encode.

In [101]:
# Show the first 5 rows before encoding the remaining text columns
data.head()


,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,23,1,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,20,1,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,21,0,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,23,1,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,19,1,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


### Separate the remaining categorical columns
Pulling out all columns that are still text (object/category type) into their own DataFrame, so we can encode them together.

In [102]:
# Select only the text (categorical) columns from data
cat_col = data.select_dtypes(include=["object", "category"])

C:\Users\Muhammad Shan\AppData\Local\Temp\ipykernel_7244\1091711355.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_col = data.select_dtypes(include=["object", "category"])


### Preview the categorical columns
Looking at the text columns that still need to be converted to numbers.

In [103]:
# Display the categorical columns DataFrame
cat_col


,part_time_job,diet_quality,parental_education_level,internet_quality,extracurricular_participation
0,No,Fair,Master,Average,Yes
1,No,Good,High School,Average,No
2,No,Poor,High School,Poor,No
3,No,Poor,Master,Good,Yes
4,No,Fair,Master,Good,No
...,...,...,...,...,...
995,No,Fair,High School,Good,Yes
996,Yes,Poor,High School,Average,Yes
997,No,Good,Bachelor,Good,Yes
998,Yes,Fair,Bachelor,Average,No


### Set up One-Hot Encoding
One-Hot Encoding turns each category into its own 0/1 column (e.g. `diet_quality_Good`, `diet_quality_Poor`), instead of assigning arbitrary numbers, since these categories don't have a natural order.

In [104]:
# Import the OneHotEncoder tool from scikit-learn
from sklearn.preprocessing import OneHotEncoder

# Create the encoder (ignore unknown categories, output a normal array instead of sparse matrix)
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

### Apply One-Hot Encoding
Fitting the encoder on the categorical columns and transforming them into numeric (0/1) values.

In [105]:
# Learn the categories and convert them into 0/1 encoded values
encoded_cat = encoder.fit_transform(cat_col)

### Preview the encoded array
Looking at the raw numeric array produced by the encoder.

In [106]:
# Display the one-hot encoded array
encoded_cat

array([[1., 0., 1., ..., 0., 0., 1.],
       [1., 0., 0., ..., 0., 1., 0.],
       [1., 0., 0., ..., 1., 1., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 1.],
       [0., 1., 1., ..., 0., 1., 0.],
       [1., 0., 0., ..., 0., 1., 0.]], shape=(1000, 13))

### Get the new column names
Getting the proper column names for the encoded columns (e.g. `diet_quality_Good`), since the encoder only gives back a plain array of numbers.

In [107]:
# Get the generated column names for the encoded columns
encoder.get_feature_names_out()

array(['part_time_job_No', 'part_time_job_Yes', 'diet_quality_Fair',
       'diet_quality_Good', 'diet_quality_Poor',
       'parental_education_level_Bachelor',
       'parental_education_level_High School',
       'parental_education_level_Master', 'internet_quality_Average',
       'internet_quality_Good', 'internet_quality_Poor',
       'extracurricular_participation_No',
       'extracurricular_participation_Yes'], dtype=object)

### Turn the encoded array into a DataFrame
Converting the plain numeric array into a proper DataFrame with column names, so it can be merged with the rest of the data later.

In [108]:
# Wrap the encoded array in a DataFrame with proper column names
encoded_cat = pd.DataFrame(encoded_cat,columns=encoder.get_feature_names_out())

### Preview the encoded DataFrame
Confirming the one-hot encoded columns look correct.

In [109]:
# Display the one-hot encoded DataFrame
encoded_cat


,part_time_job_No,part_time_job_Yes,diet_quality_Fair,diet_quality_Good,diet_quality_Poor,parental_education_level_Bachelor,parental_education_level_High School,parental_education_level_Master,internet_quality_Average,internet_quality_Good,internet_quality_Poor,extracurricular_participation_No,extracurricular_participation_Yes
0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
4,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
996,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
997,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
998,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


### Remove the original text columns
Dropping the original categorical text columns from `data`, since we now have their one-hot encoded versions separately. This leaves only the numeric columns.

In [110]:
# Drop the original text columns, keeping only numeric columns
numeric_data = data.drop(columns=cat_col.columns)

### Combine numeric and encoded columns
Joining the numeric columns and the one-hot encoded columns side by side into one final DataFrame, ready for modeling. Indexes are reset first so the rows line up correctly.

In [111]:
# Combine numeric_data and encoded_cat side by side into one final DataFrame
final_data = pd.concat([numeric_data.reset_index(drop=True),encoded_cat.reset_index(drop=True)],axis=1)

### Preview the final combined data
Checking that all columns (numeric + encoded) are present and everything is numeric.

In [112]:
# Show the first 5 rows of the final combined dataset
final_data.head()

,age,gender,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,exercise_frequency,mental_health_rating,exam_score,...,diet_quality_Good,diet_quality_Poor,parental_education_level_Bachelor,parental_education_level_High School,parental_education_level_Master,internet_quality_Average,internet_quality_Good,internet_quality_Poor,extracurricular_participation_No,extracurricular_participation_Yes
0,23,1,0.0,1.2,1.1,85.0,8.0,6,8,56.2,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1,20,1,6.9,2.8,2.3,97.3,4.6,6,8,100.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,21,0,1.4,3.1,1.3,94.8,8.0,1,1,34.3,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
3,23,1,1.0,3.9,1.0,71.0,9.2,4,1,26.8,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
4,19,1,5.0,4.4,0.5,90.9,4.9,3,1,66.4,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0


### Check final dataset size
Confirming the total number of rows and columns in the final dataset before splitting into features and target.

In [113]:
# Show the shape (rows, columns) of the final dataset
final_data.shape

(1000, 23)

### Split into features (X) and target (y)
`X` contains everything the model will use to make a prediction (all columns except exam_score). `y` contains only what we want to predict (exam_score).

In [114]:
# X = all feature columns (everything except exam_score)
X = final_data.drop(columns="exam_score")
# y = the target column we want to predict (exam_score)
y = final_data["exam_score"]

### Split into training and testing sets
Splitting the data so the model learns from one part (training set) and gets evaluated on data it has never seen (testing set). This gives an honest measure of how well it performs. 30% of the data is held out for testing, and `random_state=42` makes the split reproducible.

In [115]:
# Import the train_test_split function
from sklearn.model_selection import train_test_split
# Split X and y into training (70%) and testing (30%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

### Create the Linear Regression model
Setting up an (untrained) Linear Regression model, which will later learn the relationship between the features and exam_score.

In [116]:
# Import the LinearRegression model class
from sklearn.linear_model import LinearRegression
# Create an instance of the Linear Regression model
lr = LinearRegression()

### Train the model
Fitting the model on the training data so it learns the relationship between the features (study hours, sleep, etc) and the actual exam scores.

In [117]:
# Train the model using the training features and training target
lr.fit(X_train,y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](22,)","[ 0.07,-0.11, 9.59,...,-0.01, 0.04,-0.04]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](22,)","['age','gender','study_hours_per_day',...,'internet_quality_Poor', 'extracurricular_participation_No','extracurricular_participation_Yes']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,5.931
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,22
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(17)


### Make predictions on the test set
Using the trained model to predict exam scores for the test data (data the model has never seen before).

In [121]:
# Predict exam scores for the test set features
y_pred = lr.predict(X_test)

### Import evaluation tools
Importing the metrics we'll use to measure how good the model's predictions are, and numpy for the square root calculation used in RMSE.

In [122]:
# Import MAE, MSE, and R2 score functions from scikit-learn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Import numpy for math operations like square root
import numpy as np

### Mean Absolute Error (MAE)
MAE is the average difference between the actual and predicted exam scores, ignoring whether the model overestimated or underestimated. Lower is better.

In [125]:
# Calculate the average absolute difference between actual and predicted scores
mae = mean_absolute_error(y_test, y_pred)
# Display the MAE value
mae


4.266682796556188

### Mean Squared Error (MSE)
MSE squares each error before averaging, which punishes bigger mistakes more heavily. Its units are 'squared marks', so it's mainly used as a stepping stone to RMSE.

In [126]:
# Calculate the average squared difference between actual and predicted scores
mse = mean_squared_error(y_test, y_pred)
# Display the MSE value
mse

28.605942666899185

### Root Mean Squared Error (RMSE)
RMSE takes the square root of MSE to bring the error back into the original units (marks), making it easier to interpret than MSE.

In [127]:
# Take the square root of MSE to get RMSE in original score units
rmse = np.sqrt(mse)
# Display the RMSE value
rmse

np.float64(5.348452361842553)

### R² Score
R² shows what percentage of the variation in exam scores the model was able to explain using the features. A score close to 1 means a very good fit.

In [130]:
# Calculate how much of the variation in exam scores the model explains
r2 = r2_score(y_test, y_pred)
# Display the R2 score
r2

0.8970125439807719

### Compare actual vs predicted scores
Building a table with the real exam scores next to the model's predicted scores, so we can visually see how close the predictions are.

In [131]:
# Create a DataFrame comparing actual scores to predicted scores
model_comparison = pd.DataFrame({"Actual": y_test.reset_index(drop=True), "Predicted": y_pred})

### Preview the comparison table
Looking at the first few rows of actual vs predicted exam scores.

In [132]:
# Show the first 5 rows of the actual vs predicted comparison
model_comparison.head()

,Actual,Predicted
0,64.2,66.660546
1,72.7,75.122941
2,79.0,78.064888
3,79.5,73.694428
4,58.2,60.841841


## Summary

In this notebook, the `student_habits_performance.csv` dataset was cleaned (missing values filled, duplicates checked, ID column removed), and all categorical columns were encoded into numeric form (`gender` via manual mapping, the rest via One-Hot Encoding). The data was then split into features (`X`) and target (`y`), and further split into training and testing sets (70/30).

A **Linear Regression** model was trained on the training data and used to predict `exam_score` on the unseen test data.

**Model performance on the test set:**
- **MAE:** ~4.27 marks average error
- **MSE:** ~28.61 (squared error)
- **RMSE:** ~5.35 marks average error
- **R² Score:** ~0.897 (the model explains about 89.7% of the variation in exam scores)

Overall, the model performs well, with predicted exam scores closely matching actual exam scores in most cases, as shown in the actual vs predicted comparison table.